In [1]:
import json
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder

all_chunks = []
with open("../data/processed/kb_chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        all_chunks.append(json.loads(line))
print("Loaded chunks:", len(all_chunks))

def load_pairs(name):
    pairs = []
    with open(f"../data/processed/qa_pairs_{name}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            pairs.append(json.loads(line))
    return pairs

train_pairs = load_pairs("train")
dev_pairs = load_pairs("dev")
test_pairs = load_pairs("test")
print(f"Pairs: train={len(train_pairs)}, dev={len(dev_pairs)}, test={len(test_pairs)}")

retriever = SentenceTransformer("../data/indexes/retriever_finetuned_v1")
reranker = CrossEncoder("../data/indexes/reranker_finetuned_v1")
print("Retriever device:", retriever.device)

chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["chunk_id"] for c in all_chunks]
chunk_text_by_id = {c["chunk_id"]: c["text"] for c in all_chunks}
chunk_embeddings = retriever.encode(
    chunk_texts, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print("Chunk embeddings shape:", chunk_embeddings.shape)

Loaded chunks: 4781
Pairs: train=19640, dev=2461, test=2468


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retriever device: cuda:0


Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Chunk embeddings shape: (4781, 384)


In [2]:
from datasets import load_dataset

In [3]:
import random
random.seed(1)
sample_pairs = random.sample(train_pairs, 1500)

def batch_score_topk_check(pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20, check_k=3):
    queries = [p["query"] for p in pairs]
    query_embeddings = retriever.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i, p in enumerate(pairs):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([p["query"], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    results = []
    pos = 0
    for i, p in enumerate(pairs):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        order = np.argsort(-scores)
        ranked_candidates = [candidates[j] for j in order]
        ranked_scores = [float(scores[j]) for j in order]

        gold_set = set(p["gold_chunk_ids"])
        top1_correct = ranked_candidates[0] in gold_set
        topk_correct = any(cid in gold_set for cid in ranked_candidates[:check_k])
        top1_score = ranked_scores[0]
        margin = ranked_scores[0] - ranked_scores[1]  # gap between best and second-best

        results.append({
            "top1_score": top1_score, "margin": margin,
            "top1_correct": top1_correct, "topk_correct": topk_correct,
        })
    return results

results2 = batch_score_topk_check(sample_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, check_k=3)

topk_correct_n = sum(r["topk_correct"] for r in results2)
print(f"Top-1 correct: {sum(r['top1_correct'] for r in results2)} ({sum(r['top1_correct'] for r in results2)/len(results2):.1%})")
print(f"Gold in top-3: {topk_correct_n} ({topk_correct_n/len(results2):.1%})")

correct_margin = [r["margin"] for r in results2 if r["top1_correct"]]
wrong_margin = [r["margin"] for r in results2 if not r["top1_correct"]]
print()
print(f"Margin (top1 - top2 score) when CORRECT: mean={np.mean(correct_margin):.3f}, median={np.median(correct_margin):.3f}")
print(f"Margin (top1 - top2 score) when WRONG:   mean={np.mean(wrong_margin):.3f}, median={np.median(wrong_margin):.3f}")

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Top-1 correct: 445 (29.7%)
Gold in top-3: 698 (46.5%)

Margin (top1 - top2 score) when CORRECT: mean=1.306, median=0.945
Margin (top1 - top2 score) when WRONG:   mean=0.654, median=0.263


In [4]:
def build_triage_features(pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20, answer_k=3):
    queries = [p["query"] for p in pairs]
    query_embeddings = retriever.encode(queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i in range(len(pairs)):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([queries[i], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    examples = []
    pos = 0
    for i, p in enumerate(pairs):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        retriever_top1_sim = float(sims[i][top_idx[i][0]])

        order = np.argsort(-scores)
        ranked_candidates = [candidates[j] for j in order]
        ranked_scores = [float(scores[j]) for j in order]

        gold_set = set(p["gold_chunk_ids"])
        gold_in_topk = any(cid in gold_set for cid in ranked_candidates[:answer_k])

        examples.append({
            "query": p["query"],
            "domain": p["domain"],
            "retriever_top1_sim": retriever_top1_sim,
            "rerank_top1_score": ranked_scores[0],
            "rerank_top3_mean": float(np.mean(ranked_scores[:3])),
            "margin": ranked_scores[0] - ranked_scores[1],
            "label": "ANSWER" if gold_in_topk else "ESCALATE",
        })
    return examples

triage_train_answer_escalate = build_triage_features(train_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)
triage_dev_answer_escalate = build_triage_features(dev_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

from collections import Counter
print("Train label distribution:", Counter(e["label"] for e in triage_train_answer_escalate))
print("Dev label distribution:", Counter(e["label"] for e in triage_dev_answer_escalate))
print()
print("Example:", triage_train_answer_escalate[0])

Batches:   0%|          | 0/307 [00:00<?, ?it/s]

Batches:   0%|          | 0/6138 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Batches:   0%|          | 0/770 [00:00<?, ?it/s]

Train label distribution: Counter({'ESCALATE': 10196, 'ANSWER': 9444})
Dev label distribution: Counter({'ESCALATE': 1287, 'ANSWER': 1174})

Example: {'query': 'user: Hello, I forgot o update my address, can you help me with that?', 'domain': 'dmv', 'retriever_top1_sim': 0.5655285716056824, 'rerank_top1_score': 0.04153377190232277, 'rerank_top3_mean': -0.41183123861749965, 'margin': 0.11474770680069923, 'label': 'ANSWER'}


In [5]:
from datasets import load_dataset

squad_data = load_dataset("rajpurkar/squad")
print(squad_data)

sample = squad_data["train"][0]
print("\nFields:", sample.keys())
for k, v in sample.items():
    print(f"\n{k}:")
    print(str(v)[:300])

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Fields: dict_keys(['id', 'title', 'context', 'question', 'answers'])

id:
5733be284776f41900661182

title:
University_of_Notre_Dame

context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is 

question:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

answers:
{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [6]:
import random
random.seed(7)

squad_questions = list(set(squad_data["train"]["question"]))  # dedupe (SQuAD has repeats across paragraphs of the same article)
sampled_reject_questions = random.sample(squad_questions, 10000)

print("Sampled SQuAD questions:", len(sampled_reject_questions))
print("Example:", sampled_reject_questions[0])

Sampled SQuAD questions: 10000
Example: What besides benches and tabletops was usually created using the indirect method?


In [7]:
def build_reject_features(questions, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id, k=20):
    query_embeddings = retriever.encode(questions, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    sims = query_embeddings @ chunk_embeddings.T
    top_idx = np.argsort(-sims, axis=1)[:, :k]

    flat_pairs = []
    counts = []
    for i in range(len(questions)):
        candidates = [chunk_ids[idx] for idx in top_idx[i]]
        counts.append(len(candidates))
        for cid in candidates:
            flat_pairs.append([questions[i], chunk_text_by_id[cid]])

    all_scores = reranker.predict(flat_pairs, batch_size=64, show_progress_bar=True)

    examples = []
    pos = 0
    for i, q in enumerate(questions):
        n = counts[i]
        scores = all_scores[pos:pos+n]
        pos += n
        retriever_top1_sim = float(sims[i][top_idx[i][0]])
        order = np.argsort(-scores)
        ranked_scores = [float(scores[j]) for j in order]

        examples.append({
            "query": q,
            "domain": "out_of_domain",
            "retriever_top1_sim": retriever_top1_sim,
            "rerank_top1_score": ranked_scores[0],
            "rerank_top3_mean": float(np.mean(ranked_scores[:3])),
            "margin": ranked_scores[0] - ranked_scores[1],
            "label": "REJECT",
        })
    return examples

# split the 10k SQuAD sample into train/dev the same 90/10-ish way, to match our other splits
random.shuffle(sampled_reject_questions)
split_point = int(0.9 * len(sampled_reject_questions))
squad_train_q = sampled_reject_questions[:split_point]
squad_dev_q = sampled_reject_questions[split_point:]

triage_train_reject = build_reject_features(squad_train_q, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)
triage_dev_reject = build_reject_features(squad_dev_q, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

print("Train REJECT examples:", len(triage_train_reject))
print("Dev REJECT examples:", len(triage_dev_reject))

# sanity check: compare REJECT confidence distribution to ANSWER/ESCALATE from earlier
reject_scores = [e["rerank_top1_score"] for e in triage_train_reject]
answer_scores = [e["rerank_top1_score"] for e in triage_train_answer_escalate if e["label"] == "ANSWER"]
escalate_scores = [e["rerank_top1_score"] for e in triage_train_answer_escalate if e["label"] == "ESCALATE"]

print(f"\nREJECT   top1_score: mean={np.mean(reject_scores):.3f}, median={np.median(reject_scores):.3f}")
print(f"ANSWER   top1_score: mean={np.mean(answer_scores):.3f}, median={np.median(answer_scores):.3f}")
print(f"ESCALATE top1_score: mean={np.mean(escalate_scores):.3f}, median={np.median(escalate_scores):.3f}")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Batches:   0%|          | 0/2813 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Train REJECT examples: 9000
Dev REJECT examples: 1000

REJECT   top1_score: mean=-0.953, median=-1.017
ANSWER   top1_score: mean=0.352, median=0.300
ESCALATE top1_score: mean=-0.225, median=-0.273


In [8]:
triage_train_all = triage_train_answer_escalate + triage_train_reject
triage_dev_all = triage_dev_answer_escalate + triage_dev_reject

random.shuffle(triage_train_all)
random.shuffle(triage_dev_all)

print("Train label distribution:", Counter(e["label"] for e in triage_train_all))
print("Dev label distribution:", Counter(e["label"] for e in triage_dev_all))

with open("../data/processed/triage_train.jsonl", "w", encoding="utf-8") as f:
    for e in triage_train_all:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

with open("../data/processed/triage_dev.jsonl", "w", encoding="utf-8") as f:
    for e in triage_dev_all:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

print("Saved triage_train.jsonl and triage_dev.jsonl")

Train label distribution: Counter({'ESCALATE': 10196, 'ANSWER': 9444, 'REJECT': 9000})
Dev label distribution: Counter({'ESCALATE': 1287, 'ANSWER': 1174, 'REJECT': 1000})
Saved triage_train.jsonl and triage_dev.jsonl


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = ["retriever_top1_sim", "rerank_top1_score", "rerank_top3_mean", "margin"]
label_order = ["REJECT", "ESCALATE", "ANSWER"]

def to_xy(examples):
    X = np.array([[e[c] for c in feature_cols] for e in examples])
    y = np.array([e["label"] for e in examples])
    return X, y

X_train, y_train = to_xy(triage_train_all)
X_dev, y_dev = to_xy(triage_dev_all)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_dev_scaled)

print(classification_report(y_dev, y_pred, labels=label_order))
print("Confusion matrix (rows=true, cols=predicted), order:", label_order)
print(confusion_matrix(y_dev, y_pred, labels=label_order))

              precision    recall  f1-score   support

      REJECT       0.91      0.97      0.94      1000
    ESCALATE       0.60      0.61      0.60      1287
      ANSWER       0.60      0.55      0.57      1174

    accuracy                           0.69      3461
   macro avg       0.70      0.71      0.71      3461
weighted avg       0.69      0.69      0.69      3461

Confusion matrix (rows=true, cols=predicted), order: ['REJECT', 'ESCALATE', 'ANSWER']
[[970  29   1]
 [ 66 789 432]
 [ 26 504 644]]


In [10]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train_scaled, y_train)

y_pred_gb = gb_clf.predict(X_dev_scaled)

print(classification_report(y_dev, y_pred_gb, labels=label_order))
print("Confusion matrix (rows=true, cols=predicted), order:", label_order)
print(confusion_matrix(y_dev, y_pred_gb, labels=label_order))

              precision    recall  f1-score   support

      REJECT       0.92      0.96      0.94      1000
    ESCALATE       0.59      0.62      0.61      1287
      ANSWER       0.59      0.53      0.56      1174

    accuracy                           0.69      3461
   macro avg       0.70      0.70      0.70      3461
weighted avg       0.69      0.69      0.69      3461

Confusion matrix (rows=true, cols=predicted), order: ['REJECT', 'ESCALATE', 'ANSWER']
[[956  40   4]
 [ 54 803 430]
 [ 24 524 626]]


In [11]:
from transformers import AutoTokenizer
from datasets import Dataset as HFDataset

label_to_id = {"REJECT": 0, "ESCALATE": 1, "ANSWER": 2}
id_to_label = {v: k for k, v in label_to_id.items()}

def to_hf_dataset(examples):
    return HFDataset.from_dict({
        "text": [e["query"] for e in examples],
        "label": [label_to_id[e["label"]] for e in examples],
    })

triage_hf_train = to_hf_dataset(triage_train_all)
triage_hf_dev = to_hf_dataset(triage_dev_all)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

triage_hf_train = triage_hf_train.map(tokenize_fn, batched=True)
triage_hf_dev = triage_hf_dev.map(tokenize_fn, batched=True)

print("Train examples:", len(triage_hf_train))
print("Dev examples:", len(triage_hf_dev))
print("Example tokenized keys:", triage_hf_train[0].keys())

Map:   0%|          | 0/28640 [00:00<?, ? examples/s]

Map:   0%|          | 0/3461 [00:00<?, ? examples/s]

Train examples: 28640
Dev examples: 3461
Example tokenized keys: dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [12]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3, id2label=id_to_label, label2id=label_to_id
)
print("Model device before training:", next(model.parameters()).device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="../data/indexes/triage_classifier_v1",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=triage_hf_train,
    eval_dataset=triage_hf_dev,
    compute_metrics=compute_metrics,
)

trainer.train()

model.save_pretrained("../data/indexes/triage_classifier_v1")
tokenizer.save_pretrained("../data/indexes/triage_classifier_v1")
print("Triage classifier saved to data/indexes/triage_classifier_v1")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model device before training: cpu


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.400558,0.397688,0.800058,0.812495
2,0.355594,0.387226,0.808148,0.817432
3,0.299116,0.394624,0.815371,0.826415


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Triage classifier saved to data/indexes/triage_classifier_v1


In [13]:
print("Model device after training:", next(model.parameters()).device)

Model device after training: cuda:0


In [14]:
predictions = trainer.predict(triage_hf_dev)
y_pred_text = np.argmax(predictions.predictions, axis=1)
y_true_text = predictions.label_ids

pred_labels = [id_to_label[p] for p in y_pred_text]
true_labels = [id_to_label[t] for t in y_true_text]

print(classification_report(true_labels, pred_labels, labels=label_order))
print("Confusion matrix (rows=true, cols=predicted), order:", label_order)
print(confusion_matrix(true_labels, pred_labels, labels=label_order))

              precision    recall  f1-score   support

      REJECT       1.00      1.00      1.00      1000
    ESCALATE       0.75      0.76      0.75      1287
      ANSWER       0.73      0.72      0.73      1174

    accuracy                           0.82      3461
   macro avg       0.83      0.83      0.83      3461
weighted avg       0.82      0.82      0.82      3461

Confusion matrix (rows=true, cols=predicted), order: ['REJECT', 'ESCALATE', 'ANSWER']
[[1000    0    0]
 [   0  976  311]
 [   0  328  846]]


In [15]:
def build_hf_dataset_with_features(examples, scaler, feature_cols):
    feats = scaler.transform(np.array([[e[c] for c in feature_cols] for e in examples]))
    return HFDataset.from_dict({
        "text": [e["query"] for e in examples],
        "label": [label_to_id[e["label"]] for e in examples],
        "numeric_features": feats.tolist(),
    })

triage_hf_train_feat = build_hf_dataset_with_features(triage_train_all, scaler, feature_cols)
triage_hf_dev_feat = build_hf_dataset_with_features(triage_dev_all, scaler, feature_cols)

triage_hf_train_feat = triage_hf_train_feat.map(tokenize_fn, batched=True)
triage_hf_dev_feat = triage_hf_dev_feat.map(tokenize_fn, batched=True)

print(triage_hf_train_feat[0].keys())
print("Example numeric_features:", triage_hf_train_feat[0]["numeric_features"])

Map:   0%|          | 0/28640 [00:00<?, ? examples/s]

Map:   0%|          | 0/3461 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'numeric_features', 'input_ids', 'token_type_ids', 'attention_mask'])
Example numeric_features: [-0.842248887430207, -0.5693512376820227, -0.2227179271035727, -0.7665192276051394]


In [16]:
import torch
import torch.nn as nn
from transformers import DistilBertModel

class TriageClassifierWithFeatures(nn.Module):
    def __init__(self, num_labels=3, num_numeric_features=4):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        hidden_size = self.bert.config.dim
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size + num_numeric_features, num_labels)

    def forward(self, input_ids=None, attention_mask=None, numeric_features=None, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]  # [CLS] token representation
        cls_output = self.dropout(cls_output)
        combined = torch.cat([cls_output, numeric_features], dim=1)
        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

combined_model = TriageClassifierWithFeatures()
print("Model created. Params:", sum(p.numel() for p in combined_model.parameters()))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model created. Params: 66365199


In [17]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

base_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def custom_collate_fn(batch):
    text_fields = [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch]
    padded = base_collator(text_fields)
    padded["numeric_features"] = torch.tensor([b["numeric_features"] for b in batch], dtype=torch.float32)
    padded["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return padded

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

combined_model = TriageClassifierWithFeatures()  # fresh, untrained instance

training_args = TrainingArguments(
    output_dir="../data/indexes/triage_classifier_combined_v1",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=5,           # keep all 5, so we can compare and pick ourselves
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   # changed: minimize loss, not maximize accuracy
    greater_is_better=False,
    logging_steps=200,
    learning_rate=2e-5,
)

trainer_combined = Trainer(
    model=combined_model,
    args=training_args,
    train_dataset=triage_hf_train_feat,
    eval_dataset=triage_hf_dev_feat,
    data_collator=custom_collate_fn,
    compute_metrics=compute_metrics,
)

trainer_combined.train()
print("Best checkpoint (by eval_loss):", trainer_combined.state.best_model_checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.395770,0.387228,0.805836,0.817576
2,0.352709,0.381225,0.813349,0.821454
3,0.281229,0.392188,0.815371,0.825265
4,0.225027,0.425458,0.817972,0.829179
5,0.171965,0.498968,0.816527,0.827178


Best checkpoint (by eval_loss): ../data/indexes/triage_classifier_combined_v1\checkpoint-1790


In [18]:
predictions_combined = trainer_combined.predict(triage_hf_dev_feat)
y_pred_combined = np.argmax(predictions_combined.predictions, axis=1)
y_true_combined = triage_hf_dev_feat["label"]

pred_labels_c = [id_to_label[p] for p in y_pred_combined]
true_labels_c = [id_to_label[t] for t in y_true_combined]

print("Combined model (text + numeric features), best checkpoint:")
print(classification_report(true_labels_c, pred_labels_c, labels=label_order))
print("Confusion matrix (rows=true, cols=predicted), order:", label_order)
print(confusion_matrix(true_labels_c, pred_labels_c, labels=label_order))

Combined model (text + numeric features), best checkpoint:
              precision    recall  f1-score   support

      REJECT       1.00      1.00      1.00      1000
    ESCALATE       0.71      0.84      0.77      1287
      ANSWER       0.78      0.63      0.69      1174

    accuracy                           0.81      3461
   macro avg       0.83      0.82      0.82      3461
weighted avg       0.82      0.81      0.81      3461

Confusion matrix (rows=true, cols=predicted), order: ['REJECT', 'ESCALATE', 'ANSWER']
[[1000    0    0]
 [   0 1081  206]
 [   0  440  734]]


In [19]:
# ANSWER/ESCALATE test examples, from the untouched test_pairs split
triage_test_answer_escalate = build_triage_features(test_pairs, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

# REJECT test examples, from SQuAD's *validation* split (never touched — train split was used for REJECT train/dev)
squad_val_questions = list(set(squad_data["validation"]["question"]))
random.seed(99)
sampled_reject_test_questions = random.sample(squad_val_questions, 1000)
triage_test_reject = build_reject_features(sampled_reject_test_questions, retriever, reranker, chunk_embeddings, chunk_ids, chunk_text_by_id)

triage_test_all = triage_test_answer_escalate + triage_test_reject
random.shuffle(triage_test_all)

print("Test label distribution:", Counter(e["label"] for e in triage_test_all))

with open("../data/processed/triage_test.jsonl", "w", encoding="utf-8") as f:
    for e in triage_test_all:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
print("Saved triage_test.jsonl")

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Batches:   0%|          | 0/772 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Test label distribution: Counter({'ESCALATE': 1304, 'ANSWER': 1164, 'REJECT': 1000})
Saved triage_test.jsonl


In [20]:
triage_hf_test_feat = build_hf_dataset_with_features(triage_test_all, scaler, feature_cols)
triage_hf_test_feat = triage_hf_test_feat.map(tokenize_fn, batched=True)

predictions_test = trainer_combined.predict(triage_hf_test_feat)
y_pred_test = np.argmax(predictions_test.predictions, axis=1)
y_true_test = triage_hf_test_feat["label"]

pred_labels_test = [id_to_label[p] for p in y_pred_test]
true_labels_test = [id_to_label[t] for t in y_true_test]

print("Combined model — FINAL evaluation on held-out TEST set:")
print(classification_report(true_labels_test, pred_labels_test, labels=label_order))
print("Confusion matrix (rows=true, cols=predicted), order:", label_order)
print(confusion_matrix(true_labels_test, pred_labels_test, labels=label_order))

Map:   0%|          | 0/3468 [00:00<?, ? examples/s]

Combined model — FINAL evaluation on held-out TEST set:
              precision    recall  f1-score   support

      REJECT       1.00      1.00      1.00      1000
    ESCALATE       0.70      0.85      0.76      1304
      ANSWER       0.77      0.59      0.67      1164

    accuracy                           0.80      3468
   macro avg       0.82      0.81      0.81      3468
weighted avg       0.81      0.80      0.80      3468

Confusion matrix (rows=true, cols=predicted), order: ['REJECT', 'ESCALATE', 'ANSWER']
[[ 999    0    1]
 [   0 1104  200]
 [   0  479  685]]
